In [0]:
dbutils.widgets.text('p_table_name','salesorderdetail')
v_table_name=dbutils.widgets.get('p_table_name')
print(v_table_name)

In [0]:
%sql
USE CATALOG learn_adb_fikrat;

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

def read_cdc(table_name):
    #Fetch records since last saved lsn
    df=spark.sql(f"""select * from ms_sql_fc.cdc.vw_{table_name} 
        where start_lsn > coalesce(cdc.get_last_lsn('{table_name}'),0)""")
    df.write.mode('append').saveAsTable(f'bronze.{table_name}')
    
    #Save last watermark
    dfagg=df.agg(F.max('start_lsn')\
        .alias('last_lsn'))\
        .withColumn('table_name',F.lit(table_name))
    
    deltaTable = DeltaTable.forName(spark, 'cdc.cdc_watermarks')
    (
        deltaTable.alias('t')
        .merge(
            dfagg.alias('s'),
            't.table_name = s.table_name'
        )
        .whenMatchedUpdate(set={'last_lsn': 's.last_lsn'})
        .whenNotMatchedInsert(values={'table_name': 's.table_name', 'last_lsn': 's.last_lsn'})
        .execute()
    )

In [0]:
# read_cdc('salesorderdetail')
read_cdc(v_table_name) 